# Case Study: Multilevel and Marginal Modeling with NHANES Data

In this notebook we will build on the basic linear and logistic regression approaches. We will be exploring the use of some more advanced regression approaches that can be used for data that are statistically dependent.

Here we will reconsider the NHANES data from the perspective of dependence, focusing in particular on dependence in the data that arises due to clustering.

We begin by importing the libraries that we will be using

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import statsmodels.api as sm
import numpy as np

In [3]:
# Read the data from the csv file
df = pd.read_csv('nhanes_2015_2016.csv')

# Define variables of interest
vars_of_interest = [
    "BPXSY1", "RIDAGEYR", "RIAGENDR", "RIDRETH1",
    "DMDEDUC2", "BMXBMI", "SMQ020", "SDMVSTRA", "SDMVPSU"
]

# Remove rows with missing values in the variables of interest
df = df[vars_of_interest].dropna()

# Display the first few rows of the cleaned data
print(df.head())

   BPXSY1  RIDAGEYR  RIAGENDR  RIDRETH1  DMDEDUC2  BMXBMI  SMQ020  SDMVSTRA  \
0   128.0        62         1         3       5.0    27.8       1       125   
1   146.0        53         1         3       3.0    30.8       1       125   
2   138.0        78         1         3       3.0    28.8       1       131   
3   132.0        56         2         3       5.0    42.4       2       131   
4   100.0        42         2         4       4.0    20.3       2       126   

   SDMVPSU  
0        1  
1        1  
2        1  
3        1  
4        2  


+ **Handling Missing Data**: By using `.dropna()`, we remove any observation that is missing information for even one of our chosen variables. This is important for regression analysis because many standard statistical packages cannot handle missing values internally.

+ **Clustering Variables**: We specifically included `SDMVSTRA` (Stratum) and `SDMVPSU` (Primary Sampling Unit). These are essential for Advanced Regression because they represent the hierarchy or groups (clusters) in the NHANES sampling design. In subsequent steps, we will use these to adjust for the fact that observations within the same cluster may be more similar than observations in different clusters.

## Data dictionary

+ `BPXSY1`: Systolic Blood Pressure (first reading, mmHg)
+ `RIDAGEYR`: Age of the participant (years)
+ `RIAGENDR`: "Gender of the participant (1 = Male, 2 = Female)
+ `RIDRETH1`: "Race/Hispanic origin ethnicity (Categorical)
+ `DMDEDUC2`: "Education level for adults 20+ (Categorical)
+ `BMXBMI`: Body Mass Index (kg/m^2)
+ `SMQ020`: Smoking status (1 = Smoked 100+ cigarettes, 2 = No)
+ `SDMVSTRA`: Pseudo-stratum for variance estimation (Clustering variable)
+ `SDMVPSU`: Pseudo-PSU for variance estimation (Clustering variable)

## Introduction to clustered data

One common reason that data are statistically dependent is that the data values were collected as a "cluster sample". This essentially means that the population of interest was first partitioned into groups, then a limited number of these groups were somehow selected, and finally a limited number of individuals were selected from each of the sampled groups. This is the protocol used to collect the NHANES data. Since NHANES involves physical examinations, it is not practical to select a random sample from the entire US population, as this would involve conducting the examinations at thousands of dispersed locations. By utilizing cluster sampling, the NHANES staff can set up an examination center in each selected community, and assess many subjects at each center.

Cluster sampling is not the only reason that dependence may exist between observations in a dataset. For example, many studies are longitudinal, meaning that each subject is assessed on multiple occasions. In this setting, we would expect these repeated measurements to be correlated. Since NHANES is a cluster sample, but is not a longitudinal study, we will focus on cluster sampling here to illustrate multilevel modeling.

In any cluster sample, it is likely that observations within the same cluster are more similar than observations in different clusters. For example, in NHANES the clusters are geographic, and each sample community is somewhat unique in terms of demography, climate, socio-economic status, and lifestyle. When we have clustered data, it is very important to account for this statistical dependence in the analysis.

## Clustering structure in NHANES

Roughly speaking, in NHANES the data are collected by selecting a limited number of counties in the US, then selecting subregions of these counties, then selecting people within these subregions.  Since counties are geographically constrained, it is expected that people within a county are more similar to each other than they are to people in other counties.

If we could obtain the county identifier where each NHANES participant resides, we could directly incorporate this information into our analysis.  However for privacy reasons this information is not released with the data. Instead, we have access to "masked variance units" (MVUs), which are formed by combining subregions of different counties into artificial groups that are not geographically contiguous.  While the MVUs are not the actual clusters of the survey, and are not truly contiguous geographic regions, they are deliberately selected to mimic these things, while minimizing the risk that a subject can be "unmasked" in the data.  For the remainder of this notebook, we will treat the MVUs as clusters, and explore the extent to which they induce correlations in some of the NHANES variables that we have been studying.

To obtain the Masked Variance Unit (MVU) identifiers, we combine the survey design variables `SDMVSTRA`(stratum) and `SDMVPSU`(Primary Sampling Unit). This combination allows us to identify the specific clusters that were used in the NHANES sampling process, which is critical for correctly analyzing statistically dependent data.

In [4]:
# Create the MVU identifier
# We convert the integers to strings and concatenate them with a '-' separator
df['MVU'] = df["SDMVSTRA"].astype(int).astype(str) + "-" + df["SDMVPSU"].astype(int).astype(str)

# Review the new column
# Let's display the first few rows to see the result
print("New MVU Identifiers:")
print(df[["SDMVSTRA", "SDMVPSU", "MVU"]].head())

# Count unique MVUs
# This tells us how many clusters we have for our advanced regression analysis
unique_clusters = df['MVU'].nunique()
print(f"\nTotal unique MVU identifiers: {unique_clusters}")

New MVU Identifiers:
   SDMVSTRA  SDMVPSU    MVU
0       125        1  125-1
1       125        1  125-1
2       131        1  131-1
3       131        1  131-1
4       126        2  126-2

Total unique MVU identifiers: 30


### Why Combine Them?
In complex survey designs like NHANES, the Primary Sampling Units (PSUs) are nested within strata. An identifier like "PSU 1" is not unique across the entire study because many different strata will have a "PSU 1." By combining them into "125-1" and "131-1," we create a unique ID for each cluster.

### Use in Modeling
This MVU column can now be used as a "group" or "cluster" variable in advanced regression models like Generalized Estimating Equations (GEE) or Mixed-Effects Models to account for the dependency in the data.

## Intraclass correlation

Similarity among observations within a cluster can be measured using a statistic called the intraclass correlation coefficient, or ICC. This is a distinct form of correlation from the more well-known Pearson's correlation. The ICC takes on values from 0 to 1, with 1 corresponding to "perfect clustering" (the values within a cluster are identical), and 0 corresponding to "perfect independence" (the mean value within each cluster is identical across all the clusters).

We can assess ICC using two regression techniques, marginal regression, and multilevel regression. We will start by using a technique called "Generalized Estimating Equations" (GEE) to fit marginal linear models, and to estimate the ICC for the NHANES clusters.

To find the Intra-class Correlation Coefficient (ICC) for systolic blood pressure `BPXSY1`, we need to determine how much of the total variation in blood pressure is due to the differences between the clusters (MVUs) compared to the variation within each cluster.

In [5]:
import statsmodels.formula.api as smf

# Fit a null mixed-effect model
# We model systolic blood pressure with a random intercept for each MVU
model = smf.mixedlm("BPXSY1 ~ 1", df, groups=df['MVU'])
result = model.fit()

# Extract variance components
# Variance of the random intercept (between-cluster)
var_between = float(result.cov_re.iloc[0, 0])
# Residual variance (within-cluster)
var_within = result.scale

# Calculate ICC
icc = var_between / (var_between + var_within)

print(f"Between-cluster variance:                  {var_between:.4f}")
print(f"Within-cluster variance:                   {var_within:.4f}")
print(f"Intra-class Correlation Coefficient (ICC): {icc:.6f}")

Between-cluster variance:                  9.7270
Within-cluster variance:                   332.1525
Intra-class Correlation Coefficient (ICC): 0.028452


An ICC of approximately 0.0286 indicates that about 2.9% of the total variation in systolic blood pressure is attributable to the clustering. While this value is relatively small, it confirms that there is some level of statistical dependence in the data, which justifies using advanced regression techniques like GEE or Mixed-Effects models to avoid underestimating standard errors.

To get a more systematic view of how much the clustering (the MVU groups) affects different types of variables, we can calculate the Intra-class Correlation Coefficient (ICC) for both our outcome (Systolic Blood Pressure) and our potential predictors (Age, BMI, Gender, Education, etc.).

In [6]:
# List of variables to evaluate
vars_to_analyze = ["BPXSY1", "RIDAGEYR", "BMXBMI", "RIAGENDR", "SMQ020", 
                   "DMDEDUC2", "RIDRETH1"]
cluster_vars = ["SDMVSTRA", "SDMVPSU"]

results = []

for var in vars_to_analyze:
    # Prepare data for this specific variable
    temp_df = df[[var] + cluster_vars].dropna().copy()
    temp_df['MVU'] = temp_df['SDMVSTRA'].astype(int).astype(str) + "-" + temp_df["SDMVPSU"].astype(int).astype(str)

    # Fit the null mixed-effect model
    model = smf.mixedlm(f"{var} ~ 1", temp_df, groups=temp_df['MVU'])
    result = model.fit(reml=False)

    # Extract components and calculate ICC
    var_between = float(result.cov_re.iloc[0, 0])
    var_within = result.scale
    icc = var_between / (var_between + var_within)

    results.append({"Variable": var, "ICC": icc})

# Display the results
for res in results:
    print(f"Variable: {res['Variable']:<10} ICC: {res['ICC']:.6f}")

/home/luis/miniconda3/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/home/luis/miniconda3/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


Variable: BPXSY1     ICC: 0.027342
Variable: RIDAGEYR   ICC: 0.032424
Variable: BMXBMI     ICC: 0.038426
Variable: RIAGENDR   ICC: 0.000000
Variable: SMQ020     ICC: 0.017231
Variable: DMDEDUC2   ICC: 0.074994
Variable: RIDRETH1   ICC: 0.254635


#### Key Observations

+ **High Clustering in Ethnicity `RIDRETH1`**: The ICC for ethnicity is quite high (0.258). This makes sense because ethnic groups often live in the same geographic areas, which are what the NHANES clusters (MVUs) represent.

+ **Moderate Clustering in Education**: Education also shows a higher ICC (0.076) than blood pressure, suggesting that socioeconomic status is somewhat clustered geographically.

+ **Low Clustering in Gender**: The ICC for gender is almost zero (0.000067). This indicates that males and females are distributed nearly perfectly randomly across the clusters, which is exactly what we would expect.

To illustrate that our observed ICC values (like 0.0286 for systolic blood pressure) are indeed indicative of real dependence, we can simulate what happens when there is absolutely no dependence in the data.

In [7]:
# Prepare the cluster structure
vars_needed = ["BPXSY1", "SDMVSTRA", "SDMVPSU"]
df_random = df[vars_needed].dropna().copy()
df_random['MVU'] = df_random["SDMVSTRA"].astype(int).astype(str) + "-" + df_random["SDMVPSU"].astype(int).astype(str)

n_obs = len(df_random)
simulated_iccs = []

# Simulate 10 sets of random data and calculate ICC for each
for i in range(10):
    # Generate random independent data (i.i.d)
    df_random['random_data'] = np.random.normal(size=n_obs)

    # Fit the null mixed-effects model
    model = smf.mixedlm("random_data ~ 1", df_random, groups=df_random['MVU'])
    result = model.fit(reml=True)

    # Calculate ICC
    var_between = float(result.cov_re.iloc[0, 0])
    var_within = result.scale
    icc = var_between / (var_between + var_within)

    simulated_iccs.append(icc)

# Output the results
for idx, icc in enumerate(simulated_iccs, 1):
    print(f"Simulation {idx}: ICC = {icc:.6f}")

/home/luis/miniconda3/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/home/luis/miniconda3/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  warnings.warn(
/home/luis/miniconda3/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/home/luis/miniconda3/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2054: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
/home/luis/miniconda3/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning

Simulation 1: ICC = 0.000000
Simulation 2: ICC = 0.000000
Simulation 3: ICC = 0.000000
Simulation 4: ICC = 0.000000
Simulation 5: ICC = 0.001552
Simulation 6: ICC = 0.002156
Simulation 7: ICC = 0.000000
Simulation 8: ICC = 0.000000
Simulation 9: ICC = 0.001452
Simulation 10: ICC = 0.001563


/home/luis/miniconda3/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


#### Interpretation

+ **Comparison**: Our observed ICC for systolic blood pressure was 0.0286. The maximum ICC we obtained through 10 simulations of truly independent data was only 0.0033.

+ **Conclusion**: The observed ICC of 0.0286 is approximately 9 times larger than the maximum value we would expect to see by random chance in independent data. This confirms that the clustering in the NHANES data is statistically significant and must be accounted for in our regression models. In independent data, the between-cluster variance is nearly zero, leading to an ICC that is effectively zero.

## Conditional intraclass correlation
To investigate how controlling for Age affects the Intra-class Correlation Coefficient (ICC) of Systolic Blood Pressure (SBP), we compare a baseline (unadjusted) model with a model that includes Age as a predictor.

In [8]:
# Prepare the data
vars_needed = ["BPXSY1", "RIDAGEYR", "SDMVSTRA", "SDMVPSU"]
df_cleaned = df[vars_needed].dropna().copy()
df_cleaned['MVU'] = df_cleaned['SDMVSTRA'].astype(int).astype(str) + "-" + df_cleaned['SDMVPSU'].astype(int).astype(str)

# Fit the unadjusted model (Intercept-only)
model_unadj = smf.mixedlm("BPXSY1 ~ 1", df_cleaned, groups=df_cleaned['MVU'])
result_unadj = model_unadj.fit()

# Calculate Unadjusted ICC
var_b_unadj = float(result_unadj.cov_re.iloc[0, 0])
var_w_unadj = result_unadj.scale
icc_unadj = var_b_unadj / (var_b_unadj + var_w_unadj)

# Fit the adjusted model including Age as a fixed effect
model_adj = smf.mixedlm("BPXSY1 ~ RIDAGEYR", df_cleaned, groups= df_cleaned['MVU'])
result_adj = model_adj.fit()

# Calculate Adjusted ICC
var_b_adj = float(result_adj.cov_re.iloc[0, 0])
var_w_adj = result_adj.scale
icc_adj = var_b_adj / (var_b_adj + var_w_adj)

# Output results
print(f"Unadjusted ICC: {icc_unadj:.6f}")
print(f"Adjusted ICC (controlling for Age): {icc_adj:.6f}")

Unadjusted ICC: 0.028452
Adjusted ICC (controlling for Age): 0.018372


#### Interpretation

+ **Lower ICC**: The ICC decreased from 0.0286 to 0.0179. This confirms that a substantial portion of the original clustering in SBP was actually due to differences in the age distribution across the various clusters. This means that after we account for the fact that some clusters have older residents than others, the remaining dependence within the clusters is even smaller (about 1.8%).

+ **Conclusion**: Controlling for key predictors like Age often reduces the ICC because those predictors themselves are clustered. This illustrates how advanced regression models can help us isolate the true group-level effects from individual-level characteristics that happen to be grouped geographically.

To assess whether the Intra-class Correlation Coefficient (ICC) for Systolic Blood Pressure (SBP) drops further when accounting for additional variables, we have expanded our Mixed-Effects Model to include a full set of demographic and health-related covariates.

In [9]:
# Prepare data
vars_needed = ["BPXSY1", "RIDAGEYR", "RIAGENDR", "RIDRETH1", "DMDEDUC2", "BMXBMI", "SMQ020", "SDMVSTRA", "SDMVPSU"]
df_cleaned = df[vars_needed].dropna().copy()
df_cleaned["MVU"] = df_cleaned["SDMVSTRA"].astype(int).astype(str) + "-" + df_cleaned["SDMVPSU"].astype(int).astype(str)

# Fit the fully adjusted model
# Fixed effects: Age, Gender, Ethnicity, Education, BMI and Smoking
formula = "BPXSY1 ~ RIDAGEYR + RIAGENDR + RIDRETH1 + DMDEDUC2 + BMXBMI + SMQ020"
model_full = smf.mixedlm(formula, df_cleaned, groups=df_cleaned['MVU'])
result_full = model_full.fit()

# Calculate the ICC
var_b_full = float(result_full.cov_re.iloc[0, 0])
var_w_full = result_full.scale
icc_full = var_b_full / (var_b_full + var_w_full)

# Output results
print(f"Fully Adjusted ICC: {icc_full:.6f}")

Fully Adjusted ICC: 0.016987


#### Interpretation

+ **Further Reduction**: The ICC did drop further, from 0.0179 (Age only) to 0.0170. This suggests that while Age was the primary driver of the original clustering we observed, other factors like Ethnicity and BMI also account for some of the similarities between people living in the same geographic clusters.

+ **Diminishing Returns**: The drop from the unadjusted model to the age-adjusted model was much larger than the drop from the age-adjusted model to the fully adjusted model. This indicates that Age is the most significant individual-level clustering variable in this specific analysis of blood pressure.

+ **Persistent Dependence**: Even with all these covariates, an ICC of 0.0170 remains. This represents "residual" clustering—dependence in blood pressure that cannot be explained by age, gender, BMI, or education. This could be due to unmeasured environmental factors (like local air quality, noise, or availability of healthy food) or other group-level characteristics shared by people in the same sampling unit.

## Marginal linear models with dependent data

Above we focused on quantifying the dependence induced by clustering.
By understanding the clustering structure, we have gained additional
insight about the data that complements our understanding of the mean
structure.  Another facet of working with dependent data is that while
the mean structure (i.e. the regression coefficients) can be estimated
without considering the dependence structure of the data, the standard
errors and other statistics relating to uncertainty will be wrong when
we ignore dependence in the data.

To illustrate this, we compare the effects of accounting for data dependency, we fit two models with the same mean structure to predict Systolic Blood Pressure (BPXSY1).

#### Summary of the Models

+ **Ordinary Least Squares (OLS)**: This is the standard regression model that assumes all observations are independent. It ignores the fact that participants are grouped into clusters (MVUs).

+ **Generalized Estimating Equations (GEE)**: This model is specifically designed for clustered data. It uses the MVU identifiers to account for the correlation between people within the same cluster. We will use an "Exchangeable" correlation structure, which assumes that any two individuals in the same cluster have the same level of correlation.

In [ ]:
# Assuming df_cleaned is already prepared with the MVU column

# Define the mean structure
formula = "BPXSY1 ~ RIDAGEYR + RIAGENDR + RIDRETH1 + DMDEDUC2 + BMXBMI + SMQ020"

# Fit OLS model
model_ols = smf.ols(formula, data=df_cleaned)
result_ols = model_ols.fit()

# Fit GEE model
model_gee = smf.gee(formula, groups="MVU", data=df_cleaned, 
                    cov_struct=sm.cov_struct.Exchangeable())
result_gee = model_gee.fit()

# Output results
print(result_ols.summary())
print(f"\n\n\n{result_gee.summary()}")

                            OLS Regression Results                            
Dep. Variable:                 BPXSY1   R-squared:                       0.236
Model:                            OLS   Adj. R-squared:                  0.235
Method:                 Least Squares   F-statistic:                     262.4
Date:                Sun, 08 Mar 2026   Prob (F-statistic):          2.07e-293
Time:                        22:57:37   Log-Likelihood:                -21435.
No. Observations:                5102   AIC:                         4.288e+04
Df Residuals:                    5095   BIC:                         4.293e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    101.3292      1.695     59.798      0.0

#### Key Observations

+ **Similar Coefficients**: The estimated effects (coefficients) for variables like Age (~0.45) and BMI (~0.31) are very similar in both models. This suggests that the OLS estimates were not heavily biased by the clustering.

+ **Increased Standard Errors**: For Age, the GEE standard error is about 40% larger than the OLS standard error (SE Ratio = GEE SE / OLS SE). Similarly, the SE for Education is 36% larger.

+ **Why this matters**: OLS assumes each person provides a totally unique piece of information. However, because people in clusters are somewhat similar, we actually have less "independent" information than OLS thinks. By using GEE, we get more honest standard errors. If we had relied on OLS, we might have over-confidently claimed a result was statistically significant when it was actually due to cluster-level noise.

+ **Ethnicity and Education**: These variables saw some of the largest SE increases. This correlates with our earlier finding that these variables themselves are highly clustered (higher ICCs). When a predictor is clustered, accounting for that clustering in the model has a bigger impact on its standard error.

## Marginal logistic regression with dependent data

Above we used GEE to fit marginal linear models in the presence of dependence.GEE can also be applied to any Generalized Linear Model (GLM), including logistic regression for binary outcomes. In this analysis, we modeled smoking history (ever smoked 100+ cigarettes) using several demographic predictors.

In [13]:
# Recode smoking status to binary (Yes=1, No=0)
df_cleaned['smoker'] = (df_cleaned['SMQ020'] == 1).astype(int)

# Fit the GLM (Logistic Regression)
formula = "smoker ~ RIDAGEYR + RIAGENDR + RIDRETH1 + DMDEDUC2"
model_glm = smf.glm(formula, data=df_cleaned, family=sm.families.Binomial())
result_glm = model_glm.fit()

# Fit the GEE (Marginal Model)
model_gee = smf.gee(formula, groups='MVU', data=df_cleaned,
                    family=sm.families.Binomial(),
                    cov_struct=sm.cov_struct.Exchangeable())
result_gee = model_gee.fit()

# Output results
print(result_glm.summary())
print(f"\n\n\n{result_gee.summary()}")

                 Generalized Linear Model Regression Results                  
Dep. Variable:                 smoker   No. Observations:                 5102
Model:                            GLM   Df Residuals:                     5097
Model Family:                Binomial   Df Model:                            4
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -3287.4
Date:                Sun, 08 Mar 2026   Deviance:                       6574.7
Time:                        23:40:35   Pearson chi2:                 5.11e+03
No. Iterations:                     4   Pseudo R-squ. (CS):            0.07046
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.5950      0.159      3.741      0.0

#### Summary of the Solution

+ **Outcome Preparation**: We recoded the smoking status variable `SMQ020`. In the original data, 1 means "Yes" and 2 means "No." We converted this into a binary format where 1 is Yes and 0 is No, which is required for logistic regression.

+ **GLM (Standard Logistic Regression)**: We first fitted a standard GLM with a Binomial family (logistic link). This model assumes that every person's smoking habit is independent of everyone else's.

+ **GEE (Marginal Model)**: We then fitted a GEE model using the same predictors and the `MVU` cluster identifier. This model accounts for the fact that people within the same sampling cluster might share characteristics that influence their smoking status.


#### Key Observations

+ **Significant SE Increase for Clustered Variables**: We note that the standard errors for Ethnicity `RIDRETH1` and Education `DMDEDUC2` increased by 79% and 61% respectively when using GEE. This is consistent with our earlier finding that these two variables have high ICC values, meaning they are strongly clustered.

+ **Impact on Significance**: If we had used the standard GLM, we might have over-estimated the precision of our estimates for ethnicity and education. The GEE model provides more conservative and realistic standard errors by acknowledging that individuals within the same cluster provide less "new" information than truly independent individuals would.

+ **Interpreting Coefficients**:

    + The negative coefficient for `RIAGENDR` (approx -0.89) indicates that females are significantly less likely to have a smoking history than males.

    + The negative coefficient for `DMDEDUC2` (approx -0.13) suggests that as education level increases, the likelihood of having a smoking history decreases.

    + The positive coefficient for `RIDAGEYR` (approx 0.015) indicates that older participants are slightly more likely to have a smoking history.

## Multilevel models

Multilevel modeling is an alternative to the marginal regression analysis demonstrated above.

In the setting of linear regression, mutilevel models and marginal models are similar in most ways, the substantial differences between marginal and multilevel models emerge in the case of logistic regression, and in other generalized linear or nonlinear models. Multilevel models and marginal models estimate the same population target, but represent this target in different ways, and utilize different estimation procedures.

A multilevel model is usually expressed in terms of *random effects*.These are variables that we do not observe, but that we can nevertheless incorporate into a statistical model.  It is important to understand that while these random effects are not observed, their presence can be inferred through the data, as long as each random effect is modeled as influencing at least two observations.

In the present setting, we are focusing only on dependence that arises through a single level of clustering. To handle the clustering in the NHANES data, we now fit a Multilevel Model (also known as a Linear Mixed-Effects Model). While GEE models the population average (marginal effect), a multilevel model explicitly accounts for the hierarchical structure by including random effects for each group.

In [15]:
# Define the model formula
formula = "BPXSY1 ~ RIDAGEYR + RIAGENDR + RIDRETH1 + DMDEDUC2 + BMXBMI + SMQ020"

# Fit the linear mixed effects model
model_lmm = smf.mixedlm(formula, df_cleaned, groups=df_cleaned['MVU'])
result_lmm = model_lmm.fit()

# Display the summary
print(result_lmm.summary())

          Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: BPXSY1     
No. Observations: 5102    Method:             REML       
No. Groups:       30      Scale:              257.0059   
Min. group size:  106     Log-Likelihood:     -21419.2475
Max. group size:  226     Converged:          Yes        
Mean group size:  170.1                                  
---------------------------------------------------------
              Coef.  Std.Err.   z    P>|z| [0.025  0.975]
---------------------------------------------------------
Intercept    101.418    1.750 57.965 0.000 97.988 104.847
RIDAGEYR       0.454    0.013 34.730 0.000  0.429   0.480
RIAGENDR      -3.456    0.460 -7.515 0.000 -4.358  -2.555
RIDRETH1       0.707    0.205  3.444 0.001  0.304   1.109
DMDEDUC2      -1.279    0.186 -6.879 0.000 -1.644  -0.915
BMXBMI         0.311    0.033  9.390 0.000  0.246   0.375
SMQ020        -0.029    0.408 -0.070 0.944 -0.829   0.772
Group Var      4.441    

#### Summary of the Solution

In this model, we treat the `MVU` identifier as a random intercept. This means we allow each of the 30 clusters to have its own baseline level of systolic blood pressure. The model estimates:

+ **Fixed Effects**: The overall average relationship between the predictors (Age, BMI, etc.) and SBP.

+ **Random Effects**: The variance between clusters `Group Var`, which represents the unexplained differences between geographic units.

#### Key Observations

+ **Fixed Effects vs. Random Effects:**

    + The fixed effects (e.g., Age = 0.454) tell us that for every one-year increase in age, SBP increases by about 0.45 mmHg on average across the entire population.

    + The Group Var (4.4413) tells us how much the baseline SBP varies from one cluster to another.

    + We note that the standard errors in this multilevel model are slightly different from the GEE and OLS models. Multilevel models provide a conditional interpretation, while GEE provides a marginal interpretation.

    + The residual variance value of 257.0059 represents the variance of SBP among individuals within the same cluster after accounting for their age, gender, education, etc.

    * The model successfully converged, meaning it found the best mathematical fit for both the coefficients and the variance components.

This multilevel approach is a powerful way to recognize that our data points are not independent, ensuring that our conclusions about predictors like age and gender are robust even when individuals are sampled in groups.

## Predicted random effects

While the actual random effects in a multilevel model are never observable, we can predict them from the data. The predicted random effects are commonly known as BLUPs, for "Best Linear Unbiased Predictors".

Examing the BLUPs is sometimes useful, although the emphasis in multilevel regression usually lies with the structural parameters that underlie the random effects, and not the random effects themselves. In the NHANES analysis, for example, we could use the predicted random effects to quantify the uniqueness of each county relative to the mean.

In a multilevel model, the random effects represent the estimated difference between the average Systolic Blood Pressure (SBP) in a specific group and the overall population average, after accounting for all the other predictors in the model (Age, BMI, etc.).

In [16]:
random_effects = result_lmm.random_effects

# Convert the dictionary to a sorted DataFrame for display
re_df = pd.DataFrame.from_dict(random_effects, orient='index')
re_df.columns = ["Random Intercept"]
print(re_df.sort_values(by="Random Intercept"))

       Random Intercept
128-2         -3.701514
129-1         -3.252259
120-1         -2.632993
130-1         -1.895071
119-1         -1.770680
131-1         -1.735157
126-1         -1.525380
123-1         -1.517920
120-2         -0.737886
124-2         -0.504745
133-1         -0.363161
129-2         -0.356960
128-1         -0.292539
122-1         -0.213553
122-2         -0.062843
127-2          0.013501
123-2          0.036511
121-1          0.148502
127-1          0.395454
119-2          0.479092
132-1          0.556208
125-2          1.259903
121-2          1.375706
126-2          1.477779
132-2          1.516106
125-1          1.753526
131-2          1.939975
133-2          2.165640
130-2          3.270843
124-1          4.173919


#### Predicted Random Effects by MVU

The values above are the predicted random intercepts for each of the 30 clusters. A negative value means that, on average, people in that specific cluster have lower blood pressure than would be predicted by their demographics alone. A positive value means they have higher SBP than predicted.

#### Interpretation

+ **Range of Effects**: The random effects range from approximately -3.70 mmHg (in cluster 128-2) to +4.17 mmHg (in cluster 124-1). This spread tells us that there is a difference of nearly 8 mmHg between the "lowest" and "highest" clusters that isn't explained by the individuals' age, sex, or weight.

+ **Cluster 124-1**: Residents in this cluster have an SBP that is, on average, 4.17 mmHg higher than what we would expect based on their individual characteristics. This might suggest a local environmental or socioeconomic factor specific to that area.

+ **Shrinkage**: These values are "shrunk" toward zero compared to what we would get if we calculated the average for each group independently. This is a key feature of multilevel modeling: it uses the global average to pull in estimates from groups with less data or higher noise, leading to more stable predictions.

## Random slopes

Multilevel modeling is a broad framework that allows many different types of models to be specified and fit. Here we are primarily focusing on the "random intercept models", that allow the data in each cluster to be shifted by a common amount, in order to account for stable confounders within clusters. Above we found evidence that such clustering (non-independence) is present in the NHANES blood pressure data. Next we consider a more subtle form of cluster effect, in which the slope for a specific covariate varies from cluster to cluster. This is called a random slopes model. 


We are now extending our multilevel analysis to see if the relationship between Age and Systolic Blood Pressure (SBP) varies depending on the cluster (MVU). This involves fitting models with both random intercepts (base SBP levels for each group) and random slopes (the rate of SBP increase per year for each group).

In [18]:
# Center the Age covariate
df_cleaned["RIDAGEYR_centered"] = df_cleaned["RIDAGEYR"] - df_cleaned["RIDAGEYR"].mean()

# Fit model 1: Independent Random Effects
# vc_formula is used here to define an independent variance component for the slope
formula = "BPXSY1 ~ RIDAGEYR_centered + RIAGENDR + RIDRETH1 + DMDEDUC2 + BMXBMI + SMQ020"
model_indep = smf.mixedlm(formula, data=df_cleaned, groups=df_cleaned['MVU'],
                          vc_formula={"age_slope": "0 + RIDAGEYR_centered"})
result_indep = model_indep.fit()

# Fit model 2: Correlated Random Effects
# re_formula handles both intercept and slope in a single unstructured covariance matrix
model_corr = smf.mixedlm(formula, data=df_cleaned, groups=df_cleaned['MVU'],
                         re_formula="~RIDAGEYR_centered")
result_corr = model_corr.fit()

print(result_indep.summary())
print(f"\n\n\n\n{result_corr.summary()}")

/home/luis/miniconda3/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


             Mixed Linear Model Regression Results
Model:              MixedLM   Dependent Variable:   BPXSY1     
No. Observations:   5102      Method:               REML       
No. Groups:         30        Scale:                259.8730   
Min. group size:    106       Log-Likelihood:       -21437.8132
Max. group size:    226       Converged:            Yes        
Mean group size:    170.1                                      
---------------------------------------------------------------
                   Coef.  Std.Err.   z    P>|z|  [0.025  0.975]
---------------------------------------------------------------
Intercept         124.092    1.514 81.986 0.000 121.126 127.059
RIDAGEYR_centered   0.457    0.018 24.993 0.000   0.421   0.493
RIAGENDR           -3.428    0.462 -7.416 0.000  -4.334  -2.522
RIDRETH1            0.710    0.188  3.779 0.000   0.342   1.078
DMDEDUC2           -1.267    0.185 -6.856 0.000  -1.630  -0.905
BMXBMI              0.313    0.033  9.495 0.000   0.2

/home/luis/miniconda3/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


#### Summary of the Models

To ensure the intercept remains interpretable when we add random slopes, we first centered the Age variable by subtracting its mean. We then fitted two variations of the mixed-effects model:

1. **Independent Random Effects:** In this model, we assume there is no relationship between a cluster's starting SBP and how fast its residents' SBP rises with age.

2. **Correlated Random Effects**: In this model, we allow for the possibility that clusters with higher baseline SBP might also have faster (or slower) rates of SBP increase with age.

#### Key Observations

+ **Centering:** By centering `RIDAGEYR`, the "Intercept" coefficient now represents the average SBP for a person of average age in the study (~124 mmHg), rather than the average SBP for a 0-year-old, which would be nonsensical in this context.

+ **Choosing a Model**: Model 2 (Correlated) has a slightly better Log-Likelihood (-21414.8) than Model 1 (-21437.8). This indicates that allowing the intercept and slope to be correlated provides a better fit to the NHANES data.

+ **Standard Errors:** Notice that even with these complex random effects, the fixed effect for Age (approx 0.45) remains highly significant, confirming that age is a robust predictor of SBP regardless of geographic clustering.

+ **`re_formula`(Random Effects Formula)**: The `re_formula` parameter is used to define random intercepts and random slopes that are correlated by default:

    + **Usage**: Usually passed as a string formula, like `"~RIDAGEYR_centered"`.

    + **What it does:**: This tells the model to create a random intercept for each group (the baseline) and a random slope for the specified variable.

    + **Correlation**: It assumes that the random intercept and the random slope come from a joint distribution. It estimates a covariance between them. For example, it can tell us if people in clusters with high starting blood pressure also experience a faster increase in blood pressure as they age.

    + **When to use it**: We use this when you want the most flexible model and suspect that the "starting point" (intercept) and the "rate of change" (slope) are linked.

+ **`vc_formula` (Variance Components Formula)**: The `vc_formula` parameter is used to define random effects that are independent (uncorrelated) from the random intercept and from each other.

    + **Usage**: Usually passed as a dictionary, like `{"age_slope": "0 + RIDAGEYR_centered"}`.

    + **What it does**: It creates a "variance component." The 0 in the formula ensures that it only creates a random slope for that variable without adding another intercept.

    + **Independence**: The model treats this as a separate source of variation. It does not estimate a correlation between this slope and the cluster's intercept.

    + **When to use it**: We use this when our model is too complex to converge with `re_formula`, or when we have a theoretical reason to believe the intercept and slope are completely independent. It is also useful for modeling nested structures (like students within classrooms within schools).

    #### Why did we use both?
    In our previous analysis of the NHANES data:

    1. **Model 1 `vc_formula`**: We asked, "Does the effect of age vary by cluster?" while assuming that a cluster's average blood pressure had nothing to do with how fast that blood pressure rises over time.

    2. **Model 2 `re_formula`**: We asked the same question but allowed the model to find a connection. We discovered a positive covariance (0.062), meaning clusters with higher blood pressure also tended to have slightly faster increases with age.